# 📖 Notebook 1: Understanding CAP Trade-offs

Before we run any code, let's build a solid mental model of the CAP theorem — what it is, why it matters, and how to think about it in system design interviews.

## Learning Objectives

By the end of this notebook, you'll understand:
- What Consistency, Availability, and Partition Tolerance mean
- Why you can only pick two (and why Partition Tolerance is always required)
- The real-world impact of choosing CP vs AP
- How to decide which trade-off fits your use case

## 🛠️ Setup

Start the infrastructure first:

```bash
cd 01-foundations/cap-theorem
docker compose up -d
```

### Visualization Tools

- **Adminer** (PostgreSQL GUI): http://localhost:8080  
  Login: System `PostgreSQL`, Server `postgres-primary`, User `demo`, Password `demo`, Database `cap_demo`
- **RedisInsight** (Redis GUI): http://localhost:5540  
  Click "Add Redis Database" → Host `redis`, Port `6379`

### Kernel Selection

Select the `.venv` kernel in VS Code's kernel picker (top-right of notebook).  
If it doesn't appear, reload the window: `Cmd+Shift+P` → "Reload Window".

In [ ]:
import psycopg2
import redis
import time
import json

# Primary database — this is where we WRITE data
PRIMARY_CONFIG = {
    "host": "localhost",
    "port": 5432,
    "database": "cap_demo",
    "user": "demo",
    "password": "demo",
    "connect_timeout": 3,
}

# Replica database — this is where we READ data (may be slightly behind)
REPLICA_CONFIG = {
    "host": "localhost",
    "port": 55433,
    "database": "cap_demo",
    "user": "demo",
    "password": "demo",
    "connect_timeout": 3,
}

# Redis — always available, but may serve stale data
REDIS_CONFIG = {
    "host": "localhost",
    "port": 6379,
    "decode_responses": True
}

def get_primary():
    return psycopg2.connect(**PRIMARY_CONFIG)

def get_replica():
    return psycopg2.connect(**REPLICA_CONFIG)

def get_redis():
    return redis.Redis(**REDIS_CONFIG)

# Test all connections
for name, connect_fn in [("PostgreSQL Primary", get_primary), ("PostgreSQL Replica", get_replica)]:
    try:
        conn = connect_fn()
        conn.close()
        print(f"✅ Connected to {name}")
    except Exception as e:
        print(f"❌ {name} failed: {e}")
        print("   Run: docker compose up -d")

try:
    r = get_redis()
    r.ping()
    print("✅ Connected to Redis")
except Exception as e:
    print(f"❌ Redis failed: {e}")
    print("   Run: docker compose up -d")

# RESET demo state so this notebook is idempotent (safe to re-run from the top)
try:
    _conn = get_primary()
    _conn.autocommit = True
    _cur = _conn.cursor()
    _cur.execute("UPDATE user_profiles SET display_name = 'Alice Johnson' WHERE username = 'alice'")
    _cur.execute("DELETE FROM seat_reservations")
    _cur.execute("UPDATE events SET available_seats = total_seats")
    _conn.close()
    print("🔄 Reset demo data (idempotent start)")
except Exception as _e:
    print(f"⚠️  Could not reset demo data: {_e}")


## 🤔 What is CAP Theorem?

Imagine you have an app that stores data across multiple servers (a **distributed system**).  
The CAP theorem says you can only guarantee **two out of three** properties:

```
                    Consistency (C)
                        /\
                       /  \
                      /    \
                     / PICK \
                    /  TWO!  \
                   /          \
    Availability (A) ──────── Partition Tolerance (P)
```

Let's define each one:

| Property | What It Means | Analogy |
|----------|--------------|----------|
| **Consistency (C)** | Every read gets the most recent write | Every library branch has the exact same books right now |
| **Availability (A)** | Every request gets a response | The library is always open, even if some books are outdated |
| **Partition Tolerance (P)** | System works even when servers can't talk to each other | The library stays open even when the phone line between branches is down |

⚠️ **Important**: Consistency in CAP is different from ACID consistency in databases. CAP consistency means "all nodes see the same data at the same time."

## 🔑 The Key Insight: Partition Tolerance is NOT Optional

In any real distributed system, **network failures WILL happen**. Servers crash, cables get cut, data centers lose power. You MUST handle partitions.

This simplifies CAP from "pick 2 of 3" to a single choice:

```
┌─────────────────────────────────────────────────────────────┐
│                                                             │
│   Since Partition Tolerance is REQUIRED, you really choose: │
│                                                             │
│     CP = Consistency + Partition Tolerance                   │
│          → Reject requests when you can't guarantee          │
│            the data is up-to-date                            │
│                                                             │
│     AP = Availability + Partition Tolerance                  │
│          → Always respond, even if data might be stale       │
│                                                             │
└─────────────────────────────────────────────────────────────┘
```

💡 **Interview tip**: When an interviewer asks about CAP, say: *"Since partition tolerance is non-negotiable in a distributed system, the real question is whether we prioritize consistency or availability."

## 🔁 Beyond CAP: PACELC

CAP only describes what happens **during a network partition**. But what about the
99% of the time when there is no partition? That's where **PACELC** comes in
(pronounced "pass-elk"), proposed by Daniel Abadi in 2012:

> **If** there is a **P**artition, choose between **A**vailability and **C**onsistency.
> **E**lse (no partition), choose between **L**atency and **C**onsistency.

```
                ┌─ Partition? ──────────────────┐
                │                               │
              YES                              NO
                │                               │
        Pick A or C                      Pick L or C
        (classic CAP)                    (everyday trade-off)
```

This is more useful in practice because **most systems are not partitioned most of the time** —
the everyday trade-off is between *latency* (fast reads from a nearby replica) and
*consistency* (slower reads from the primary that are always fresh).

| System | During partition | Normally | PACELC label |
|--------|------------------|----------|--------------|
| Google Spanner | Consistency | Consistency | **PC/EC** |
| PostgreSQL (single primary) | Consistency | Consistency | **PC/EC** |
| Amazon DynamoDB (default) | Availability | Latency | **PA/EL** |
| Cassandra | Availability | Latency | **PA/EL** |
| MongoDB (default) | Consistency | Latency | **PC/EL** |

💡 **In interviews**, mentioning PACELC shows you understand that the consistency-vs-latency
trade-off is a daily reality, not just a partition-time concern.


## 🌍 A Concrete Example

Imagine a website with two servers — one in the USA, one in Europe. A user in the USA updates their display name.

**Normal operation** (no partition):
```
User A (USA) updates name → USA Server → replicates to → Europe Server
User B (Europe) reads name → Europe Server → sees updated name ✅
```

**During a network partition** (servers can't communicate):
```
User A (USA) updates name → USA Server → ✗ can't reach → Europe Server
User B (Europe) reads name → Europe Server → ???
```

Now what happens when User B reads? That depends on our choice:

| Choice | What Happens | User Experience |
|--------|--------------|-----------------|
| **CP** | Return an error — we can't guarantee freshness | "Service temporarily unavailable" |
| **AP** | Return the old name — stale but available | User B sees the old name for a while |

Let's see this in action with our database setup!

In [ ]:
# Let's demonstrate the normal flow: write to primary, read from replica

# Step 1: Read the current state of Alice's profile
conn = get_primary()
cursor = conn.cursor()
cursor.execute("SELECT display_name, bio FROM user_profiles WHERE username = 'alice'")
row = cursor.fetchone()
conn.close()

print("📦 Alice's profile (read from PRIMARY):")
print(f"   Display Name: {row[0]}")
print(f"   Bio: {row[1]}")
print()

# Step 2: Read from the REPLICA — should be the same
conn = get_replica()
cursor = conn.cursor()
cursor.execute("SELECT display_name, bio FROM user_profiles WHERE username = 'alice'")
row_replica = cursor.fetchone()
conn.close()

print("📦 Alice's profile (read from REPLICA):")
print(f"   Display Name: {row_replica[0]}")
print(f"   Bio: {row_replica[1]}")
print()

if row == row_replica:
    print("✅ Both servers return the SAME data — the system is consistent!")
else:
    print("⚠️  Data differs — replication lag detected (eventual consistency)")

In [ ]:
# Now let's write to the primary and check the replica immediately
# This shows that replication has a small delay (even without a partition!)

# Write to PRIMARY
conn = get_primary()
conn.autocommit = True
cursor = conn.cursor()
cursor.execute(
    "UPDATE user_profiles SET display_name = 'Alice J. (Updated!)', updated_at = NOW() WHERE username = 'alice'"
)
conn.close()
print("✏️  Updated Alice's name on PRIMARY to: 'Alice J. (Updated!)'")

# Immediately read from REPLICA — is it there yet?
checks = []
for delay_ms in [0, 10, 50, 100, 500]:
    time.sleep(delay_ms / 1000)
    conn = get_replica()
    cursor = conn.cursor()
    cursor.execute("SELECT display_name FROM user_profiles WHERE username = 'alice'")
    name = cursor.fetchone()[0]
    conn.close()
    is_updated = "Updated" in name
    checks.append((delay_ms, name, is_updated))

print()
print("⏱️  Checking REPLICA at different delays after write:")
print(f"   {'Delay':>8}  {'Name on Replica':<30}  {'In Sync?'}")
print("   " + "-" * 55)
for delay_ms, name, synced in checks:
    icon = "✅" if synced else "⏳"
    print(f"   {delay_ms:>5} ms  {name:<30}  {icon}")

print()
print("💡 Even in a healthy system, replication is not instant.")
print("   This tiny gap is 'replication lag' — it's the foundation of eventual consistency.")

In [ ]:
# Reset Alice's name back to original
conn = get_primary()
conn.autocommit = True
cursor = conn.cursor()
cursor.execute(
    "UPDATE user_profiles SET display_name = 'Alice Johnson', updated_at = NOW() WHERE username = 'alice'"
)
conn.close()
time.sleep(0.5)  # wait for replication
print("🔄 Reset Alice's name back to 'Alice Johnson'")

## 🎫 When to Choose Consistency (CP)

Some systems **cannot tolerate stale reads** — showing wrong data causes real harm:

### Example: Ticket Booking
Imagine seat 6A on a flight. If two users book it at the same time during a partition, you get a **double-booking**. Someone shows up at the airport with no seat!

### Example: Bank Transfers  
If your balance shows $5000 on one server but $0 on another (because a transfer hasn't replicated), you could overdraw your account.

**The rule**: If showing wrong data causes **money loss, safety issues, or broken contracts** → choose **CP**.

### ❌ Bad practice → ✅ Best practice: enforcing the rule

A naïve booking implementation might look like this:

```python
# ❌ BAD: race condition between SELECT and INSERT
seats_left = db.query("SELECT available_seats FROM events WHERE id = 1")
if seats_left > 0:
    db.execute("INSERT INTO seat_reservations ...")   # two users can pass the check at the same time!
```

The fix is to push the consistency rule **into the database itself** so it's
impossible to violate, no matter how many concurrent requests arrive:

```sql
-- ✅ GOOD: a UNIQUE constraint guarantees no two reservations
-- can share the same (event_id, seat_number).
UNIQUE(event_id, seat_number)
```

Now the second `INSERT` always fails with a `UniqueViolation`, even under
massive concurrency. The next code cell demonstrates exactly that.


In [ ]:
# Let's demonstrate why ticket booking NEEDS consistency
# We'll simulate two users trying to book the same seat

print("🎫 Scenario: Two users race to book seat A1 at Taylor Swift concert")
print("=" * 65)
print()

# Open a connection in autocommit mode UP FRONT.
# (psycopg2 forbids changing autocommit mid-transaction, so we set it
# before running any queries.)
conn = get_primary()
conn.autocommit = True
cursor = conn.cursor()

# Check available seats
cursor.execute("SELECT name, available_seats FROM events WHERE id = 1")
event = cursor.fetchone()
print(f"🎵 Event: {event[0]}")
print(f"   Available seats: {event[1]}")
print()

# User 1 books seat A1 — the UNIQUE constraint on (event_id, seat_number)
# is what enforces "no double-booking" (CP behaviour at the database level).
try:
    cursor.execute(
        "INSERT INTO seat_reservations (event_id, seat_number, user_id) VALUES (1, 'A1', 1)"
    )
    print("✅ User 1 (Alice) booked seat A1")
except psycopg2.errors.UniqueViolation:
    print("⚠️  Seat A1 already booked")

# User 2 tries to book the SAME seat
try:
    cursor.execute(
        "INSERT INTO seat_reservations (event_id, seat_number, user_id) VALUES (1, 'A1', 2)"
    )
    print("✅ User 2 (Bob) booked seat A1")
except psycopg2.errors.UniqueViolation:
    print("❌ User 2 (Bob) REJECTED — seat A1 already taken!")

conn.close()

print()
print("💡 The UNIQUE constraint on (event_id, seat_number) enforces consistency.")
print("   This is a CP design: we reject the second booking rather than allow a double-book.")
print("   If we used an AP design (e.g. write to two replicas independently),")
print("   both users might 'successfully' book the same seat during a partition!")


## 📱 When to Choose Availability (AP)

Most systems can tolerate temporary staleness — showing slightly old data is better than showing nothing:

### Example: Social Media Profiles
If you update your profile picture, it's fine if some users see the old picture for a few minutes.

### Example: Netflix Catalog
If someone updates a movie description, showing the old description temporarily isn't a disaster.

### Example: Like Counts
If a post has 1,000 likes but one server shows 999, nobody notices or cares.

**The rule**: If stale data is **annoying but not harmful** → choose **AP**.

In [ ]:
# Let's demonstrate an AP scenario with Redis (the cache is always available)
# Even if the database is slow or unreachable, Redis responds instantly

r = get_redis()

print("📱 Scenario: Social media — viewing post like counts")
print("=" * 60)
print()

# Fetch a post from the database and cache it in Redis
conn = get_primary()
cursor = conn.cursor()
cursor.execute("SELECT id, content, like_count FROM posts ORDER BY like_count DESC LIMIT 1")
post = cursor.fetchone()
conn.close()

post_data = {"id": post[0], "content": post[1], "likes": post[2]}
r.set(f"post:{post[0]}", json.dumps(post_data), ex=60)  # cache for 60 seconds

print(f"📝 Post: '{post_data['content']}'")
print(f"   Likes (from DB): {post_data['likes']}")
print(f"   Cached in Redis with 60s TTL")
print()

# Now someone likes the post — the DB is updated but cache is stale
conn = get_primary()
conn.autocommit = True
cursor = conn.cursor()
cursor.execute(f"UPDATE posts SET like_count = like_count + 50 WHERE id = {post_data['id']}")
cursor.execute(f"SELECT like_count FROM posts WHERE id = {post_data['id']}")
new_likes = cursor.fetchone()[0]
conn.close()

# Read from cache — it's stale but FAST and AVAILABLE
cached = json.loads(r.get(f"post:{post_data['id']}"))

print(f"After 50 new likes:")
print(f"   Database says:  {new_likes} likes (accurate but slower)")
print(f"   Redis says:     {cached['likes']} likes (stale but instant)")
print(f"   Difference:     {new_likes - cached['likes']} likes behind")
print()
print("💡 This is AP behavior: Redis always responds, even with stale data.")
print("   For social media likes, this is totally fine!")
print("   The cache will update on the next TTL expiry or explicit invalidation.")

## 🧠 Decision Framework: CP or AP?

Ask yourself this one question:

> **"Would it be catastrophic if users briefly saw inconsistent data?"**

| Answer | Choice | Examples |
|--------|--------|----------|
| **Yes, catastrophic** | Choose **CP** | Bank balance, seat booking, stock price, inventory count |
| **No, just annoying** | Choose **AP** | Profile picture, like count, movie description, DNS |

### Real-World Systems: How They Choose

```
CP Systems (Consistency First)           AP Systems (Availability First)
═══════════════════════════════           ══════════════════════════════
🏦 Bank ledgers                          📱 Social media feeds
🎫 Ticket booking                        📺 Netflix catalog
📦 Amazon inventory count                ⭐ Yelp restaurant hours
📈 Stock trading order book              🌐 DNS resolution
🔐 User authentication state             📊 Analytics dashboards
```

In [ ]:
# Interactive exercise: classify these scenarios as CP or AP

scenarios = [
    ("Flight seat reservation",        "CP", "Double-booking puts two people in one seat"),
    ("Twitter follower count",          "AP", "Off by a few followers is fine"),
    ("Bank account balance",            "CP", "Wrong balance → overdraft or fraud"),
    ("YouTube video view count",        "AP", "Nobody cares if it's 999,998 vs 1,000,000"),
    ("E-commerce last item in stock",   "CP", "Overselling means cancelled orders and angry customers"),
    ("User profile bio",               "AP", "Stale bio for a few seconds is harmless"),
    ("Distributed lock / leader election", "CP", "Two leaders = split brain = data corruption"),
    ("News article comments",           "AP", "Missing a new comment for a moment is fine"),
]

print("🧠 CAP Trade-off Quiz")
print("=" * 70)
print()
print(f"{'Scenario':<35} {'Choice':>6}  Why")
print("-" * 70)
for scenario, choice, reason in scenarios:
    icon = "🔒" if choice == "CP" else "🌐"
    print(f"{scenario:<35} {icon} {choice:>3}   {reason}")

print()
print("💡 In interviews, most systems are AP. Only pick CP when stale data causes real harm.")

## 🧹 Cleanup

In [ ]:
# Clean up Redis keys and seat reservations we created
r = get_redis()
keys = r.keys("post:*")
if keys:
    r.delete(*keys)
    print(f"🧹 Cleaned up {len(keys)} Redis keys")

conn = get_primary()
conn.autocommit = True
cursor = conn.cursor()
cursor.execute("DELETE FROM seat_reservations")
cursor.execute("UPDATE events SET available_seats = total_seats")
conn.close()
print("🧹 Cleaned up seat reservations")

## 📚 Summary

### Key Takeaways

1. **CAP theorem** — in a distributed system, you can only have 2 of 3: Consistency, Availability, Partition Tolerance
2. **Partition Tolerance is mandatory** — network failures happen, so the real choice is CP vs AP
3. **CP** = reject requests during partitions to stay accurate (banks, tickets, inventory)
4. **AP** = always respond, even with stale data (social media, streaming, reviews)
5. **Most systems are AP** — only choose CP when stale data causes real harm

### Next Up

In **Notebook 2**, we'll go deeper with a **hands-on demo** of consistency vs availability — writing to a primary, reading from a replica, and simulating network partitions to see the trade-offs in real time.